# RAG with and without reranking

See how reranking changes the selected passages and whether that changes the generated answer.

## One question, two RAG paths

Compare both paths on **“What is the cause of COVID-19?”** Each path selects five passages from the same 30 candidates and uses the same prompt template and generator settings.

| Component | Model or setup |
| --- | --- |
| Embedding | `qwen/qwen3-embedding-4b` · 2,560 dimensions |
| Reranker | `qwen/qwen3-reranker-8b` |
| Generator | `qwen/qwen3-30b-a3b-instruct-2507` |
| Dataset | MS MARCO v2.1 validation (`microsoft/ms_marco`) |
| Pipeline | Haystack: query embedding → retrieve 30 passages → select five, or rerank then select five → generate answer |


## Setup and live execution

Use the project's pinned environment and dependency lock. Live execution is the default: `MODE="live"` runs query embedding, retrieval, reranking, generation and scoring. Set `OPENROUTER_API_KEY` before starting the kernel. See [setup.md](../../setup.md) for environment, kernel and launch instructions.

The index covers the frozen **152,576-passage validation subset**, not the full validation corpus. Document embedding remains stopped at this snapshot. Exact Python and package versions are specified in the repository lock.

In [1]:
STUDY = 'demo'
from pathlib import Path
import os, sys
from IPython.display import display, Markdown, HTML
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "configs/tutorial_inputs.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.live_tutorial import TutorialRun
from shared_scoring.provider import embedding_payload, rerank_payload
from shared_scoring.tutorial_views import TutorialViews
from shared_scoring.notebook_display import display_table, details_html
from shared_scoring.io import read_json
import inspect, json
import pandas as pd

MODE = os.environ.get("NOTEBOOK_MODE", "live").lower()  # live | replay | plan
RUN_ID = os.environ.get("TUTORIAL_RUN_ID") or None
RESUME = os.environ.get("TUTORIAL_RESUME") == "1"  # explicit same-run resume only

tutorial = TutorialRun(PROJECT, STUDY, mode=MODE, run_id=RUN_ID, resume=RESUME)
tutorial.preflight(create_client=False)  # assets, credentials and offline tests; no model requests
client = None
if MODE == "live":
    from openai import OpenAI
    import httpx
    # Standard OpenAI-style client; the transport records this run and enforces caps.
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=tutorial.retrieval["api_base"],
        max_retries=0,
        timeout=tutorial.retrieval["timeout_seconds"],
        http_client=httpx.Client(transport=tutorial.transport()),
    )
    tutorial.bind_client(client)
# Selecting Live authorizes this notebook's declared workflow; no second paid-call switch.

view = TutorialViews(tutorial)
view.show("configuration")
from haystack import Pipeline, component
question = tutorial.declarations["demo_question"]["question"]


## Dataset

The corpus uses MS MARCO v2.1 validation. The following row illustrates the dataset structure; the full dataset is not bundled.


In [3]:
view.show('corpus')


Validation questions: 15585
Dataset revision: a47ee7aae8d7d466ba15f9f0bfac3b3681087b3a


Inspect a question, its passages, source URLs, support annotations and reference answers. This Rachel Carson example explains the schema; the pipeline demonstration uses COVID-19.

In `is_selected`, **1** marks a passage annotated as supporting the reference answer. These dataset labels let us evaluate retrieval coverage.


In [4]:
if MODE != "plan":
    query_file = tutorial.upstream("prepared_retrieval") / "queries.parquet"
    if query_file.exists():
        schema_examples = pd.read_parquet(query_file)
        example = schema_examples[schema_examples.question.str.contains("Rachel Carson", case=False, na=False)]
        display(example.head(1) if not example.empty else schema_examples.head(1))


{'answers': ['Rachel Carson writes The Obligation to Endure because believes that as man tries to eliminate unwanted insects and weeds, however he is actually causing more problems by polluting the environment.'],
 'passages': {'is_selected': [0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
  'passage_text': ["Read to write - grow a love of books (even if it's a comic book) Organizing an argument into paragraphs, words [that] worked together for a common purpose Walker: Using models - read other research papers, look at MLA samples Maintain a larger perspective of the world and in your writing - stay objective Write about what fascinates you",
   'Carson subtly defers her writing in just the right writing technique for it to not be subject to an induction run rampant style which grabs the readers interest without biasing the whole article. Rachel slights the opposition very nicely Rachel Carson: “THE OBLIGATION TO ENDURE”. 4 by keeping it simple yet informative and persuasive and stays away from being 

Each candidate retains its passage text, source metadata, embedding rank and similarity score. The corpus contains deduplicated passages with stable IDs and a 1,800-character passage limit. The retrieval section below displays the thirty candidates after retrieval has completed.


In [5]:
@component
class RetrieveQuestion:
    @component.output_types(question=str)
    def run(self, question: str):
        if question != tutorial.questions()[0]["question"]:
            raise ValueError("Update the declared demo_question in configs/tutorial_inputs.json before changing the question.")
        tutorial.retrieve(embed_api)
        return {"question": question}


## Query embedding and retrieval

Document embeddings and the FAISS index already exist for the frozen 152,576-passage corpus. The workflow embeds the question, retrieves thirty candidates, and produces separate top-five contexts and generator responses.

In [7]:
def embed_api(questions):
    # Actual OpenAI-style embedding call. Passage vectors remain prepared inputs.
    return client.embeddings.create(**embedding_payload(
        tutorial.retrieval["embedding_model"], questions, tutorial.retrieval["dimensions"],
        instruction=tutorial.retrieval["query_instruction"]))

display({"question": question, "candidates": 30, "passages_per_answer": 5})


,Setting,Value
0,embedding_model,qwen/qwen3-embedding-4b
,Setting,Value
1,reranker_model,qwen/qwen3-reranker-8b
2,generator_model,qwen/qwen3-30b-a3b-instruct-2507
3,embedding_dimensions,2560
4,dataset_revision,a47ee7aae8d7d466ba15f9f0bfac3b3681087b3a
5,question,What is the cause of COVID-19?
6,top_candidates,30
7,top_context,5
8,temperature,0.7


The following cell shows the Haystack retrieval pipeline over the frozen **152,576 cached passages**. Inspect the thirty candidates after retrieval. Execution requires the corresponding corpus/index and matching configuration.

In [8]:
retrieval_pipeline = Pipeline()
retrieval_pipeline.add_component("retrieve", RetrieveQuestion())
if MODE != "plan":
    retrieval_pipeline.run({"retrieve": {"question": question}})
    view.show("contexts", query_id="demo-covid19")


,id,text,score,source_url,embedding_rank,truncated
0,p_3728f219848bf793cb32,The truth that we must stare down is that cancer and infections cannot be separated from each other. Cancer—always believed to be caused by genetic cell mut...,0.491919,http://drsircus.com/cancer/is-cancer-a-fungus/,1,False
,id,text,score,source_url,embedding_rank,truncated
1,p_39df7d1187fcc5489493,"Bacteria are the most common cause of infective endocarditis, especially streptococci, staphylococci, and enterococci. Although fungi and viruses can cause ...",0.470086,https://quizlet.com/144209873/unit-6-chapter-23-alterations-of-cardio-function-flash-cards/,2,False
2,p_de76223c44b3e69cd180,"California is a state, so it doesn't think! :-)) Just kidding. Cancer is caused by a genetic mutation. Genetic mutations are caused by UV light (sun, sun la...",0.469425,https://answers.yahoo.com/question/index?qid=20080803190629AAlO45U,3,False
3,p_8729b7573df5124d2792,"Hand, foot, and mouth disease is generally caused by coxsackievirus A-16, which is a member of the enterovirus family. There are other types of enteroviruse...",0.463244,https://www.emedicinehealth.com/hand_foot_and_mouth_disease/article_em.htm,4,False
4,p_2c34293f7e92f17e5467,"You can also have it and not know it. Doctors call this “walking pneumonia.” If your pneumonia is caused by a bacteria or virus, you can spread it to someon...",0.461744,https://www.webmd.com/lung/understanding-pneumonia-basics,5,False
5,p_a727a7d1be3cf3c5a664,"HIV-1, which causes almost all the cases of AIDS worldwide; HIV-2, which causes an AIDS-like illness. HIV-2 infection is uncommon in North America. What cau...",0.458251,https://www.webmd.com/hiv-aids/tc/human-immunodeficiency-virus-hiv-infection-topic-overview,6,False
6,p_36c253c659a76c80373a,"In most cases, acute pancreatitis is caused by gallstones or heavy alcohol use. Other causes include medications, autoimmune disease, infections, trauma, me...",0.456779,https://www.webmd.com/digestive-disorders/digestive-diseases-pancreatitis,7,False
7,p_0ff8dfc59276423c2b92,Streptococcal infections are caused by any one of several species of Streptococcus. Different groups of these bacteria are spread in different ways—for exam...,0.456163,http://www.merckmanuals.com/home/infections/bacterial-infections/streptococcal-infections,8,False
8,p_44a4729531569405d184,"The cause of cholera is infection by the V. cholera bacteria. These bacteria were discovered in 1883. The German bacteriologist, Robert Koch (1843-1910), st...",0.453879,https://www.medicalnewstoday.com/articles/189269.php,9,False


## Embedding RAG

Select embedding ranks 1–5 from the same thirty candidates. The Haystack pipeline connects this selection to the prompt builder and generator.

Inspect the shared prompt template, followed by this path’s five-passage context and generated answer. Both RAG paths use the same template and generator settings.

In [9]:
def generate_api(messages, settings, routing=None):
    # The actual SDK call; the tutorial invokes it for each declared question/arm.
    return client.chat.completions.create(
        model=settings["model"], messages=messages,
        temperature=settings["temperature"], max_tokens=settings["max_tokens"],
        **settings["extras"], **({"extra_body": {"provider": routing}} if routing else {}))
@component
class GenerateAnswer:
    def __init__(self, arm):
        self.arm = arm
    @component.output_types(answer=str)
    def run(self, question: str):
        if question != tutorial.questions()[0]["question"]:
            raise ValueError("Update the declared demo_question in configs/tutorial_inputs.json before changing the question.")
        tutorial.generate(generate_api, arm=self.arm)
        answers = view.answers()
        return {"answer": answers.loc[answers.arm.eq(self.arm), "answer"].iloc[0]}

embedding_pipeline = Pipeline()
embedding_pipeline.add_component("generate", GenerateAnswer("baseline"))


Original shared prompt template:
Answer the question given the context.
Question: {{ query }}
Context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}
Answer:



In [10]:
if MODE != "plan":
    display(embedding_pipeline.run({"generate": {"question": question}}))
    view.show("walkthrough", query_id="demo-covid19")


,id,text,score,source_url,embedding_rank,truncated
0,p_3728f219848bf793cb32,The truth that we must stare down is that cancer and infections cannot be separated from each other. Cancer—always believed to be caused by genetic cell mut...,0.491919,http://drsircus.com/cancer/is-cancer-a-fungus/,1,False
1,p_39df7d1187fcc5489493,"Bacteria are the most common cause of infective endocarditis, especially streptococci, staphylococci, and enterococci. Although fungi and viruses can cause ...",0.470086,https://quizlet.com/144209873/unit-6-chapter-23-alterations-of-cardio-function-flash-cards/,2,False
2,p_de76223c44b3e69cd180,"California is a state, so it doesn't think! :-)) Just kidding. Cancer is caused by a genetic mutation. Genetic mutations are caused by UV light (sun, sun la...",0.469425,https://answers.yahoo.com/question/index?qid=20080803190629AAlO45U,3,False
3,p_8729b7573df5124d2792,"Hand, foot, and mouth disease is generally caused by coxsackievirus A-16, which is a member of the enterovirus family. There are other types of enteroviruse...",0.463244,https://www.emedicinehealth.com/hand_foot_and_mouth_disease/article_em.htm,4,False
4,p_2c34293f7e92f17e5467,"You can also have it and not know it. Doctors call this “walking pneumonia.” If your pneumonia is caused by a bacteria or virus, you can spread it to someon...",0.461744,https://www.webmd.com/lung/understanding-pneumonia-basics,5,False


The cause of COVID-19 is a virus called SARS-CoV-2.

{'request_body': {'model': 'qwen/qwen3-30b-a3b-instruct-2507',
  'messages': [{'role': 'user',
    'content': "Answer the question given the context.\nQuestion: What is the cause of COVID-19?\nContext:\n\n    The truth that we must stare down is that cancer and infections cannot be separated from each other. Cancer—always believed to be caused by genetic cell mutations—can in reality be caused by infections from viruses, bacteria, and fungi.\n\n    Bacteria are the most common cause of infective endocarditis, especially streptococci, staphylococci, and enterococci. Although fungi and viruses can cause infective endocarditis, they are not the most frequent cause. Infective endocarditis is caused by infection with microorganisms, not an autoimmune response. A client develops inflammation of the endocardium caused by Staphylococcus aureus.\n\n    California is a state, so it doesn't think! :-)) Just kidding. Cancer is caused by a genetic mutation. Genetic mutations are caused by UV light 

## Embedding + reranking RAG

Rerank the thirty candidates and select the five highest-ranked passages. The Haystack pipeline connects that selection to the prompt builder and generator.

This path uses the same prompt template and generator settings as Embedding RAG.


In [11]:
def rerank_api(question, candidates):
    # OpenRouter rerank is exposed through the OpenAI client's generic POST API.
    return client.post("/rerank", cast_to=dict[str, object], body=rerank_payload(
        tutorial.retrieval["reranker_model"], question, candidates))

@component
class RerankQuestion:
    @component.output_types(question=str)
    def run(self, question: str):
        if question != tutorial.questions()[0]["question"]:
            raise ValueError("Update the declared demo_question in configs/tutorial_inputs.json before changing the question.")
        tutorial.rerank(rerank_api)
        return {"question": question}

reranked_pipeline = Pipeline()
reranked_pipeline.add_component("rerank", RerankQuestion())
reranked_pipeline.add_component("generate", GenerateAnswer("reranked"))
reranked_pipeline.connect("rerank.question", "generate.question")


Inspect the reranked five-passage context and generated answer. Expand the trace for the rendered prompt, request body, response, finish reason, token usage and measured timing.


In [12]:
if MODE != "plan":
    display(reranked_pipeline.run({"rerank": {"question": question}}))
    view.show("walkthrough", query_id="demo-covid19")


,id,text,score,source_url,embedding_rank,truncated
0,p_65af6d90ff5ebade896e,A cytokine storm is a potentially fatal immune reaction in which the cytokines continue to stimulate the production of immune cells when they are no longer ...,0.453026,http://www.wellness.com/reference/allergies/cytokines,10,False
1,p_6482a0495822ba20641b,Primary Cause of Influenza B . The primary cause of Influenza B is the result: of transmission of an infectious agent by another person by one or more of th...,0.448999,http://www.rightdiagnosis.com/i/influenza_b/causes.htm,15,False
2,p_696be041d02106a27d65,Myocarditis is an inflammatory disease of the heart muscle (myocardium) that can result from a variety of causes. While most cases are produced by a viral i...,0.447175,https://medical-dictionary.thefreedictionary.com/Inflammation+of+the+heart,16,False
3,p_d5fc92f8a485f1700c58,The cause is unknown. Researchers think that fibromyalgia is due to a problem with how the central nervous system processes pain. Possible causes or trigger...,0.439789,https://medlineplus.gov/ency/article/000427.htm,22,False
4,p_72663f8dc5ea39d41e04,This is mostly due to the fact that a C. albicans infection stems from the commensal population of C. albicans in the human microflora. Candidiasis is cause...,0.437836,https://microbewiki.kenyon.edu/index.php/Candida_albicans_(Pathogenesis),24,False


The cause of COVID-19 is the SARS-CoV-2 virus.

{'request_body': {'model': 'qwen/qwen3-30b-a3b-instruct-2507',
  'messages': [{'role': 'user',
    'content': 'Answer the question given the context.\nQuestion: What is the cause of COVID-19?\nContext:\n\n    A cytokine storm is a potentially fatal immune reaction in which the cytokines continue to stimulate the production of immune cells when they are no longer needed. The cause of this reaction remains unknown.\n\n    Primary Cause of Influenza B . The primary cause of Influenza B is the result: of transmission of an infectious agent by another person by one or more of the following: saliva, air, cough, fecal-oral route, surfaces, blood, needles, blood transfusions, sexual contact, mother to fetus, etc. Influenza B: Related Medical Conditions\n\n    Myocarditis is an inflammatory disease of the heart muscle (myocardium) that can result from a variety of causes. While most cases are produced by a viral infection, an inflammation of the heart muscle may also be instigated by toxins, dr

## What changed?

Both answers identify SARS-CoV-2, but neither selected context mentions COVID-19 or SARS-CoV-2. This prompt does not explicitly prohibit outside knowledge, so a correct answer alone does not demonstrate evidence support or a reranking benefit.

The selected passages changed; the core answer did not. This optional example motivates the evidence-only prompts and separate retrieval, reference-overlap, and faithfulness evaluations in the experiments.


In [13]:
# Score only answers produced by this RUN_ID. Live includes RAGAS where declared.
score_run = await tutorial.score()
if score_run is not None:
    display(score_run)
view.show('scores')
tutorial.show()


,rag_type,answer,names_sars_cov_2,finish_reason,output_truncated,empty_answer,context_mentions_covid_19,context_mentions_sars_cov_2,generation_successful_request_seconds
0,baseline,The cause of COVID-19 is a virus called SARS-CoV-2.,True,stop,False,False,False,False,1.204402
,rag_type,answer,names_sars_cov_2,finish_reason,output_truncated,empty_answer,context_mentions_covid_19,context_mentions_sars_cov_2,generation_successful_request_seconds
1,reranked,The cause of COVID-19 is the SARS-CoV-2 virus.,True,stop,False,False,False,False,1.396032


Both paths identify SARS-CoV-2: True
Both paths produced identical wording: False
